In [ ]:
!pip install -q --upgrade google-generativeai langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.3 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
API_KEY = userdata.get('GOOGLE_API_KEY')

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-2.0-flash",api_key=API_KEY)

In [ ]:
!pip install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.1 MB/s eta 0:00:00


In [ ]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 4.7 MB/s eta 0:00:00


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdfloader = PyPDFLoader('/content/TermsofUseVB.pdf')
loaded_pdf_doc = pdfloader.load()

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50, separators=['\n\n', '\n', ' '])

In [ ]:
chunks = recursive_splitter.split_documents(loaded_pdf_doc)
print(len(chunks))

124


In [ ]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.4/188.4 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 kB 5.3 MB/s eta 0:00:

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(model='models/embedding-001',google_api_key=API_KEY)

In [ ]:
from langchain.vectorstores import Chroma
vector_store = Chroma.from_documents(documents=chunks, embedding=embeddings)

In [ ]:
vector_index = vector_store.as_retriever(search_kwargs={'k':8})

In [ ]:
from langchain.chains import RetrievalQA
rag = RetrievalQA.from_chain_type(
    model,
    retriever = vector_index,
    return_source_documents=True
)

In [ ]:
question = 'what is spam according to the doc?'
response = rag({"query":question})
print(response)

<ipython-input-14-c448bf1da85f>:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = rag({"query":question})


{'query': 'what is spam according to the doc?', 'result': 'Spam is posting untargeted, unwanted, and repetitive content, comments, or messages with the intention to spam a Public Forum or otherwise the Platform and to drive traffic from the Platform to other third-party sites. Posting links to external websites with malware and other prohibited sites is not allowed.', 'source_documents': [Document(metadata={'source': '/content/TermsofUseVB.pdf', 'creator': 'PyPDF', 'page_label': '8', 'creationdate': '', 'page': 7, 'producer': 'Skia/PDF m136 Google Docs Renderer', 'title': 'Terms of Use - VB', 'total_pages': 16}, page_content='of\n \nindividuals,\n \ncontent\n \nor\n \ncomments\n \nuploaded\n \nin\n \norder\n \nto\n \nhumiliate\n \nsomeone,\n \nsexual\n \nharassment\n \nor\n \nsexual\n \nbullying\n \nin\n \nany\n \nform.\n 50.   51.  Spam:  Posting  untargeted,  unwanted  and  repetitive  content,  \ncomments\n \nor\n \nmessages\n \nwith\n \nan\n \nintention\n \nto'), Document(metadata=